# Model Optimization
Este notebook tem como foco realizar o treinamento de modelos de Machine Learning utilizando uma separação simples de treino e teste (Hold-out), otimização de hiperparâmetros (Hyperparameter Tuning com Scikit-Optimize) e seleção de variáveis (Feature Selection). Objetivo: prever `total_beneficios` com base em indicadores socioeconômicos.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import cross_val_score

from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import SelectFromModel

# Para tuning Bayesiano
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Carregar a base processada no nb05
df = pd.read_csv('data/base_final_v2.csv')

print(f"Shape: {df.shape}")
df.head()

Shape: (5570, 21)


,cod_municipio,municipio,populacao_total,pib_municipal,va_adm_publica,va_agropecuaria,va_industria,qtd_ben_bas,qtd_ben_var,qtd_ben_bvj,...,qtd_ben_bvg,qtd_ben_bsp,taxa_alfabetizacao,populacao_urbana_2010,pib_per_capita,taxa_urbanizacao,perc_va_agropecuaria,perc_va_industria,perc_va_adm_publica,total_beneficios
0,1100015,Alta Floresta D'Oeste - RO,22516.0,734467,173122,311469,27845,1143.4,2305.0,279.4,...,36.3,256.9,91.59,13970.0,32.619781,0.620448,0.424075,0.037912,0.235711,4051.9
1,1100023,Ariquemes - RO,111148.0,3211294,782306,293001,407675,2913.6,6766.7,805.8,...,121.0,637.8,94.08,76525.0,28.892054,0.688496,0.091241,0.126950,0.243611,11313.2
2,1100031,Cabixi - RO,5067.0,238414,46579,136659,8117,134.8,323.8,38.8,...,5.6,52.2,89.82,2693.0,47.052299,0.531478,0.573200,0.034046,0.195370,558.8
3,1100049,Cacoal - RO,86416.0,2792506,629109,341315,236801,2536.1,5171.6,624.4,...,115.6,1083.6,93.71,61921.0,32.314687,0.716546,0.122225,0.084799,0.225285,9603.0
4,1100056,Cerejeiras - RO,16088.0,743062,123365,151690,27433,494.8,1159.5,137.9,...,26.3,40.7,92.15,14419.0,46.187345,0.896258,0.204142,0.036919,0.166022,1874.9


In [3]:
# Definir a variável alvo
target = 'total_beneficios'

# Variáveis identificadoras ou de vazamento de dados (data leakage) que compõem o total
vars_remover = [
    'cod_municipio', 'municipio', target,
    'qtd_ben_bas', 'qtd_ben_var', 'qtd_ben_bvj', 
    'qtd_ben_bvn', 'qtd_ben_bvg', 'qtd_ben_bsp'
]

features = [col for col in df.columns if col not in vars_remover]

X = df[features].copy()
y = df[target].copy()

print(f"Features ({len(features)}): {features}")

# Divisão da base em 80% Treino e 20% Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Tamanho Treino: {X_train.shape[0]} | Tamanho Teste: {X_test.shape[0]}")

Features (12): ['populacao_total', 'pib_municipal', 'va_adm_publica', 'va_agropecuaria', 'va_industria', 'taxa_alfabetizacao', 'populacao_urbana_2010', 'pib_per_capita', 'taxa_urbanizacao', 'perc_va_agropecuaria', 'perc_va_industria', 'perc_va_adm_publica']
Tamanho Treino: 4456 | Tamanho Teste: 1114


In [4]:
#=========================================================
# FUNÇÕES DE OTIMIZAÇÃO DE HIPERPARÂMETROS (BAYESIANA)
#=========================================================
def tuning_random_forest(X_tun_train, y_tun_train, max_calls=15):
    print("Iniciando Tuning: RandomForestRegressor...")
    
    space = [
        Integer(10, 200, name='n_estimators'),
        Integer(2, 20, name='max_depth'),
        Integer(2, 20, name='min_samples_split'),
        Integer(1, 10, name='min_samples_leaf')
    ]

    i = 0
    @use_named_args(space)
    def objective(n_estimators, max_depth, min_samples_split, min_samples_leaf):
        nonlocal i
        i += 1
        
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42,
            n_jobs=-1
        )
        
        # Cross-validation com 3 folds
        scores = cross_val_score(
            model,
            X_tun_train,
            y_tun_train,
            cv=3,
            scoring='neg_mean_absolute_error',
            n_jobs=-1
        )
        
        print(f'interação = {i} | MAE = {scores.mean()} ')
        # Converter para MAE positivo
        return -scores.mean()
    
    res = gp_minimize(objective, space, n_calls=max_calls, random_state=42)
    
    best_params = {
        'n_estimators': res.x[0],
        'max_depth': res.x[1],
        'min_samples_split': res.x[2],
        'min_samples_leaf': res.x[3]
    }
    
    print(f"Melhores parâmetros RF: {best_params} | MAE (CV-3): {res.fun:.4f}")
    
    return RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)

def tuning_lgbm(X_tun_train, y_tun_train, max_calls=15):
    print("Iniciando Tuning: LGBMRegressor...")
    space = [
        Real(0.01, 0.3, name='learning_rate'),
        Integer(20, 200, name='n_estimators'),
        Integer(2, 12, name='max_depth'),
        Integer(2, 30, name='min_child_samples')
    ]
    
    X_tr, X_val, y_tr, y_val = train_test_split(X_tun_train, y_tun_train, test_size=0.20, random_state=42)
    
    @use_named_args(space)
    def objective(learning_rate, n_estimators, max_depth, min_child_samples):
        params = {'learning_rate': learning_rate, 'n_estimators': n_estimators, 'max_depth': max_depth, 'min_child_samples': min_child_samples}
        model = LGBMRegressor(**params, random_state=42, verbose=-1)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        return mean_absolute_error(y_val, preds)
    
    res = gp_minimize(objective, space, n_calls=max_calls, random_state=42)
    best_params = {
        'learning_rate': res.x[0],
        'n_estimators': res.x[1],
        'max_depth': res.x[2],
        'min_child_samples': res.x[3]
    }
    print(f"Melhores parâmetros LGBM: {best_params} | MAE Otimizado (Validação): {res.fun:.4f}")
    return LGBMRegressor(**best_params, random_state=42, verbose=-1)

def tuning_xgboost(X_tun_train, y_tun_train, max_calls=15):
    print("Iniciando Tuning: XGBRegressor...")
    space = [
        Real(0.01, 0.3, name='learning_rate'),
        Integer(50, 300, name='n_estimators'),
        Integer(3, 10, name='max_depth'),
        Real(0.5, 1.0, name='subsample')
    ]
    
    X_tr, X_val, y_tr, y_val = train_test_split(X_tun_train, y_tun_train, test_size=0.20, random_state=42)
    
    @use_named_args(space)
    def objective(learning_rate, n_estimators, max_depth, subsample):
        params = {'learning_rate': learning_rate, 'n_estimators': n_estimators, 'max_depth': max_depth, 'subsample': subsample}
        model = XGBRegressor(**params, random_state=42, n_jobs=-1, objective='reg:squarederror')
        model.fit(X_tr, y_tr, verbose=False)
        preds = model.predict(X_val)
        return mean_absolute_error(y_val, preds)
    
    res = gp_minimize(objective, space, n_calls=max_calls, random_state=42)
    best_params = {
        'learning_rate': res.x[0],
        'n_estimators': res.x[1],
        'max_depth': res.x[2],
        'subsample': res.x[3]
    }
    print(f"Melhores parâmetros XGB: {best_params} | MAE Otimizado (Validação): {res.fun:.4f}")
    return XGBRegressor(**best_params, random_state=42, n_jobs=-1, objective='reg:squarederror')

In [5]:
#=========================================================
# FUNÇÃO DE SELEÇÃO DE VARIÁVEIS (FEATURE SELECTION)
#=========================================================
def feature_selection(X_train, y_train, max_features=None):
    print("Executando Seleção de Features com RandomForestRegressor...")
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    selector = SelectFromModel(rf, prefit=True, max_features=max_features, threshold=-np.inf if max_features else 'mean')
    features_selecionadas = X_train.columns[selector.get_support()]
    
    print(f"Variáveis originais: {X_train.shape[1]} | Selecionadas: {len(features_selecionadas)}")
    return list(features_selecionadas)

In [6]:
#=========================================================
# FUNÇÃO PRINCIPAL DE TREINAMENTO E AVALIAÇÃO
#=========================================================
def treinar_modelo(X_train, X_test, y_train, y_test, modelo_nome='rf', optimize_hyperparams=True, fs_max_features=None):
    print(f"\n{'='*50}\nTREINANDO MODELO: {modelo_nome.upper()} (Sem K-Fold)\n{'='*50}")
    
    X_tr = X_train.copy()
    X_te = X_test.copy()
    
    # 1. Feature Selection usando apenas dados de Treino
    features_to_use = X_tr.columns.tolist()
    # if fs_max_features is not None:
    #     features_to_use = feature_selection(X_tr, y_train, max_features=fs_max_features)
    #     X_tr = X_tr[features_to_use]
    #     X_te = X_te[features_to_use]
        
    # 2. Hyperparameter Tuning (Somente com os dados de Treino)
    if optimize_hyperparams:
        if modelo_nome == 'rf':
            model = tuning_random_forest(X_tr, y_train, max_calls=15)
        elif modelo_nome == 'lgbm':
            model = tuning_lgbm(X_tr, y_train, max_calls=15)
        elif modelo_nome == 'xgb':
            model = tuning_xgboost(X_tr, y_train, max_calls=15)
        else:
            raise ValueError("Modelo não suportado.")
    else:
        # Treinar com parâmetros default se não otimizado
        if modelo_nome == 'rf': model = RandomForestRegressor(random_state=42)
        elif modelo_nome == 'lgbm': model = LGBMRegressor(random_state=42)
        elif modelo_nome == 'xgb': model = XGBRegressor(random_state=42)
        
    # 3. Treinar Modelo Final do conjunto Treino inteiro
    print(f"\nTreinando {modelo_nome.upper()} na base de Treino...")
    model.fit(X_tr, y_train)
    
    # 4. Avaliar Modelo na base de Teste de forma final
    preds = model.predict(X_te)
    rmse = root_mean_squared_error(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    print(f"\n{'='*50}\nRESULTADO FINAL NO TESTE - {modelo_nome.upper()}\n{'='*50}")
    print(f"RMSE Final: {rmse:.2f}")
    print(f"MAE Final : {mae:.2f}")
    print(f"R² Final  : {r2:.4f}")
    
    df_imp = None
    # Agregar importâncias
    if hasattr(model, 'feature_importances_'):
        df_imp = pd.Series(model.feature_importances_, index=features_to_use).sort_values(ascending=False)
        print("\nTop Variáveis Mais Importantes:")
        print(df_imp.head(10))
        
    return {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'importances': df_imp,
        'model': model
    }

In [7]:
# Testar com Random Forest (com Tuning reduzido)
res_rf = treinar_modelo(X_train, X_test, y_train, y_test, modelo_nome='rf', optimize_hyperparams=True, fs_max_features=None)


TREINANDO MODELO: RF (Sem K-Fold)
Iniciando Tuning: RandomForestRegressor...


interação = 1 | MAE = -2709.3223334896534 
interação = 2 | MAE = -3295.676363368855 
interação = 3 | MAE = -2344.6420628536166 
interação = 4 | MAE = -5276.162372943111 
interação = 5 | MAE = -5341.953354645786 
interação = 6 | MAE = -4495.9290594466565 
interação = 7 | MAE = -2597.185043131291 
interação = 8 | MAE = -2293.7256646810924 
interação = 9 | MAE = -2293.4484964428343 
interação = 10 | MAE = -2545.7957493564063 
interação = 11 | MAE = -2280.09122508167 
interação = 12 | MAE = -2276.718698768706 
interação = 13 | MAE = -2275.945180214359 
interação = 14 | MAE = -2290.6071276013386 
interação = 15 | MAE = -2281.5631073108366 
Melhores parâmetros RF: {'n_estimators': np.int64(23), 'max_depth': np.int64(18), 'min_samples_split': np.int64(11), 'min_samples_leaf': np.int64(4)} | MAE (CV-3): 2275.9452

Treinando RF na base de Treino...

RESULTADO FINAL NO TESTE - RF
RMSE Final: 10112.12
MAE Final : 2072.11
R² Final  : 0.4960

Top Variáveis Mais Importantes:
populacao_urbana_2010   

In [8]:
# Testar com LightGBM
res_lgbm = treinar_modelo(X_train, X_test, y_train, y_test, modelo_nome='lgbm', optimize_hyperparams=True, fs_max_features=None)


TREINANDO MODELO: LGBM (Sem K-Fold)
Iniciando Tuning: LGBMRegressor...
Melhores parâmetros LGBM: {'learning_rate': 0.1258362092902435, 'n_estimators': np.int64(29), 'max_depth': np.int64(11), 'min_child_samples': np.int64(9)} | MAE Otimizado (Validação): 1730.6884

Treinando LGBM na base de Treino...

RESULTADO FINAL NO TESTE - LGBM
RMSE Final: 17432.98
MAE Final : 2901.50
R² Final  : -0.4979

Top Variáveis Mais Importantes:
populacao_total          168
taxa_alfabetizacao       167
pib_per_capita            80
va_adm_publica            73
perc_va_adm_publica       69
taxa_urbanizacao          62
va_industria              58
populacao_urbana_2010     50
va_agropecuaria           41
perc_va_agropecuaria      38
dtype: int32


In [9]:
# Testar com XGBoost
res_xgb = treinar_modelo(X_train, X_test, y_train, y_test, modelo_nome='xgb', optimize_hyperparams=True, fs_max_features=None)


TREINANDO MODELO: XGB (Sem K-Fold)
Iniciando Tuning: XGBRegressor...
Melhores parâmetros XGB: {'learning_rate': 0.013385397319685235, 'n_estimators': np.int64(286), 'max_depth': np.int64(7), 'subsample': 0.7798924699238633} | MAE Otimizado (Validação): 1478.1166

Treinando XGB na base de Treino...

RESULTADO FINAL NO TESTE - XGB
RMSE Final: 16720.19
MAE Final : 2683.48
R² Final  : -0.3779

Top Variáveis Mais Importantes:
populacao_urbana_2010    0.510659
perc_va_adm_publica      0.107959
perc_va_industria        0.081093
va_adm_publica           0.066624
perc_va_agropecuaria     0.063545
va_industria             0.059757
populacao_total          0.040965
va_agropecuaria          0.031347
pib_per_capita           0.016508
taxa_alfabetizacao       0.009110
dtype: float32
